In [ ]:
# ================================================================
# SINGLE CELL — patch + reload + hls4ml conversion
# Paste at bottom of cnn_model.py notebook
# model, x_test, y_test must be in memory from training cell
# ================================================================

import os, sys, shutil, numpy as np
import hls4ml

# ── Configuration ─────────────────────────────────────────────────
HLS_PROJECT_DIR = "/home/mas47153/cnn_accelerator/hls4ml_cnn_project"
FPGA_PART       = "xc7z020clg484-1"
CLOCK_PERIOD_NS = 10
REUSE_FACTOR    = 1

# ── Step 1: Patch hls4ml files ────────────────────────────────────
print("Step 1: Patching hls4ml files...")

# Patch keras_to_hls.py
f1 = "/home/mas47153/.local/lib/python3.12/site-packages/hls4ml/converters/keras_to_hls.py"
src = open(f1).read()
changed = False
if "or 'batch_shape' in keras_layer['config']:" not in src:
    src = src.replace(
        "if 'batch_input_shape' in keras_layer['config']:",
        "if 'batch_input_shape' in keras_layer['config'] or 'batch_shape' in keras_layer['config']:"
    )
    src = src.replace(
        "input_shapes = [keras_layer['config']['batch_input_shape']]",
        "input_shapes = [keras_layer['config'].get('batch_input_shape', keras_layer['config'].get('batch_shape'))]"
    )
    open(f1, 'w').write(src)
    print("  Patched keras_to_hls.py")
    changed = True
else:
    print("  keras_to_hls.py already patched")

# Patch core.py — ALL occurrences of batch_input_shape
f2 = "/home/mas47153/.local/lib/python3.12/site-packages/hls4ml/converters/keras/core.py"
src = open(f2).read()

# Fix input_shape line
if "layer['input_shape'] = keras_layer['config']['batch_input_shape'][1:]" in src:
    src = src.replace(
        "layer['input_shape'] = keras_layer['config']['batch_input_shape'][1:]",
        "layer['input_shape'] = keras_layer['config'].get('batch_input_shape', keras_layer['config'].get('batch_shape'))[1:]"
    )
    changed = True

# Fix output_shape line
if "output_shape = keras_layer['config']['batch_input_shape']" in src:
    src = src.replace(
        "output_shape = keras_layer['config']['batch_input_shape']",
        "output_shape = keras_layer['config'].get('batch_input_shape', keras_layer['config'].get('batch_shape'))"
    )
    changed = True

# Fix dtype line
if "dtype = keras_layer['config']['dtype']" in src:
    src = src.replace(
        "dtype = keras_layer['config']['dtype']",
        "dtype = keras_layer['config'].get('dtype', 'float32')\n    if isinstance(dtype, dict):\n        dtype = dtype.get('config', {}).get('name', 'float32')"
    )
    changed = True

open(f2, 'w').write(src)
print("  Patched keras/core.py")

# Verify all lines
print("  Verification:")
for i, line in enumerate(open(f2).read().splitlines()):
    if 'batch_input_shape' in line or 'batch_shape' in line:
        print(f"    Line {i}: {line.strip()}")

# ── Step 2: Force reload hls4ml from patched files ────────────────
print("\nStep 2: Reloading hls4ml...")
to_remove = [k for k in sys.modules if 'hls4ml' in k]
for k in to_remove:
    del sys.modules[k]
import hls4ml
print(f"  Reloaded {len(to_remove)} modules — hls4ml {hls4ml.__version__}")

# ── Step 3: Build config manually ────────────────────────────────
print("\nStep 3: Building config...")
config = {
    'Model': {'Precision': 'ap_fixed<16,8>', 'ReuseFactor': REUSE_FACTOR},
    'LayerName': {}
}
for lyr in model.layers:
    n, lt = lyr.name, lyr.__class__.__name__
    if lt in ('Conv2D', 'Dense'):
        config['LayerName'][n] = {
            'Precision': {
                'weight': 'ap_fixed<16,8>',
                'bias'  : 'ap_fixed<16,8>',
                'result': 'ap_fixed<16,8>',
                'accum' : 'ap_fixed<32,16>',
            },
            'ReuseFactor': REUSE_FACTOR
        }
    elif lt == 'MaxPooling2D':
        config['LayerName'][n] = {'Precision': {'result': 'ap_fixed<16,8>'}}
    elif lt in ('Flatten', 'InputLayer'):
        config['LayerName'][n] = {'Precision': {'result': 'ap_ufixed<16,10>'}}

for n, c in config['LayerName'].items():
    p = c.get('Precision', {})
    print(f"  {n:<12}  w={p.get('weight','def'):16s}  acc={p.get('accum','def')}")

# ── Step 4: Convert to HLS ────────────────────────────────────────
print("\nStep 4: Converting to HLS...")
if os.path.exists(HLS_PROJECT_DIR):
    shutil.rmtree(HLS_PROJECT_DIR)

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config   = config,
    output_dir   = HLS_PROJECT_DIR,
    backend      = 'Vivado',
    part         = FPGA_PART,
    clock_period = CLOCK_PERIOD_NS,
    io_type      = 'io_stream'
)
hls_model.compile()
print("  Compiled OK")

# ── Step 5: Validate ──────────────────────────────────────────────
print("\nStep 5: Validating (1000 samples)...")
kp = np.argmax(model.predict(x_test[:1000], verbose=0), axis=1)
hp = np.argmax(hls_model.predict(x_test[:1000]), axis=1)
ka = np.mean(kp == y_test[:1000])
ha = np.mean(hp == y_test[:1000])
mr = np.mean(kp == hp)
print(f"  Keras  accuracy : {ka*100:.2f}%")
print(f"  hls4ml accuracy : {ha*100:.2f}%")
print(f"  Prediction match: {mr*100:.2f}%")
print("  OK" if mr >= 0.95 else "  WARNING: match below 95%")

# ── Step 6: Synthesis ─────────────────────────────────────────────
print("\nStep 6: Running synthesis...")
os.environ['PATH'] = '/opt/Xilinx/Vivado/2020.2/bin:' + os.environ['PATH']
tcl = os.path.join(HLS_PROJECT_DIR, 'build_prj.tcl')
if os.path.exists(tcl):
    lines = [l for l in open(tcl).read().splitlines()
             if 'config_array_partition' not in l]
    open(tcl, 'w').write('\n'.join(lines) + '\n')
    print("  TCL patched for Vivado 2020.2")
try:
    hls_model.build(csim=False, synth=True, vsynth=False)
    print("  Synthesis complete")
except Exception as e:
    print(f"  Synthesis error: {e}")
    print("  Source Vivado first: source /opt/Xilinx/Vivado/2020.2/settings64.sh")

# ── Step 7: Comparison table ──────────────────────────────────────
print()
print("=" * 60)
print("Resource Comparison: Custom VHDL vs hls4ml")
print("=" * 60)
vhdl = dict(DSP=199, RAMB36=2, RAMB18=1, LUT=42474, FF=6574, Fmax=102.6)
print(f"\n{'Resource':<16}{'Custom VHDL':>14}{'hls4ml':>14}{'Device Max':>14}")
print("-" * 60)
print(f"{'DSP48E1':<16}{vhdl['DSP']:>14}  {'(see rpt)':>13}  {'220':>13}")
print(f"{'RAMB36':<16}{vhdl['RAMB36']:>14}  {'(see rpt)':>13}  {'140':>13}")
print(f"{'RAMB18':<16}{vhdl['RAMB18']:>14}  {'(see rpt)':>13}  {'280':>13}")
print(f"{'LUT':<16}{vhdl['LUT']:>14}  {'(see rpt)':>13}  {'53200':>13}")
print(f"{'FF':<16}{vhdl['FF']:>14}  {'(see rpt)':>13}  {'106400':>13}")
print(f"{'Fmax MHz':<16}{vhdl['Fmax']:>14.1f}  {'(see rpt)':>13}  {'100 target':>13}")
print(f"{'Accuracy %':<16}{98.84:>14.2f}  {ha*100:>13.2f}")

rpt = os.path.join(HLS_PROJECT_DIR, 'cnn_mnist_prj', 'solution1',
                   'syn', 'report', 'cnn_mnist_csynth.rpt')
if os.path.exists(rpt):
    print("\n" + "=" * 60)
    print("HLS Synthesis Report:")
    print("=" * 60)
    try:
        print(hls4ml.report.read_vivado_report(HLS_PROJECT_DIR))
    except:
        print(open(rpt).read())
else:
    print("\n  Synthesis report not found — check Vivado PATH")